In [ ]:
# Make sure to install the required libraries first:
# pip install accelerate bitsandbytes
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig
from PIL import Image
import requests
import torch

#from huggingface_hub import login
#login(token="YOUR_TOKEN_HERE")

model_id = "google/medgemma-1.5-4b-it"

# 4-bit NF4 quantization — cuts VRAM from ~8 GB (bf16) to ~2.5 GB
# and significantly increases token throughput on consumer GPUs.
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",            # NF4 is recommended for LLMs
    bnb_4bit_compute_dtype=torch.bfloat16, # compute in bf16 for accuracy
    bnb_4bit_use_double_quant=True,        # extra ~0.4 bits/param saving
)

model = AutoModelForImageTextToText.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",         # Accelerate places layers optimally
    low_cpu_mem_usage=True,
)
processor = AutoProcessor.from_pretrained(model_id)

# Image attribution: Stillwaterising, CC0, via Wikimedia Commons
image_url = "https://upload.wikimedia.org/wikipedia/commons/c/c8/Chest_Xray_PA_3-8-2010.png"
image = Image.open(requests.get(image_url, headers={"User-Agent": "example"}, stream=True).raw)

messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": "Describe this X-ray"}
        ]
    }
]

inputs = processor.apply_chat_template(
    messages, add_generation_prompt=True, tokenize=True,
    return_dict=True, return_tensors="pt"
).to(model.device)

input_len = inputs["input_ids"].shape[-1]

with torch.inference_mode():
    generation = model.generate(**inputs, max_new_tokens=2000, do_sample=False)
    generation = generation[0][input_len:]

decoded = processor.decode(generation, skip_special_tokens=True)
print(decoded)


Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the cpu.
The image processor of type `Gemma3ImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 
Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.


This is a chest X-ray image. Here's a description of what I see:

*   **Overall Appearance:** The image shows the chest cavity, including the lungs, heart, ribs, and other structures.
*   **Lungs:** The lungs appear clear, with no obvious signs of consolidation (like pneumonia) or large masses. The lung markings are visible, indicating the normal structure of the lung tissue.
*   **Heart:** The heart size appears within normal limits.
*   **Mediastinum:** The mediastinum (the space between the lungs containing the heart, great vessels, trachea, etc.) appears normal in width and alignment.
*   **Ribs and Bones:** The ribs and other bony structures are visible and appear intact.
*   **Diaphragm:** The diaphragm (the muscle separating the chest from the abdomen) is visible at the bottom of the image.
*   **No Obvious Abnormalities:** There are no obvious signs of acute pathology, such as pneumothorax (collapsed lung), pleural effusion (fluid in the pleural space), or significant consolida